# Audio Translation with Eleven Labs Text-to-Speech

This notebook:
1. Transcribes audio to English (Whisper)
2. Translates to Telugu/Hindi (NLLB)
3. Converts translated text to speech (Eleven Labs)

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import torch
import whisper
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from pydub import AudioSegment
from pydub.silence import split_on_silence
from elevenlabs import VoiceSettings
from elevenlabs.client import ElevenLabs

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device.upper()}")
print()

In [ ]:
# ===== CONFIGURATION =====
ELEVENLABS_API_KEY = "your_api_key_here"  # Replace with your Eleven Labs API key
AUDIO_FILE = "audio_files/samanddog.wav"
OUTPUT_LANGUAGE = "hindi"  # 'telugu' or 'hindi'
OUTPUT_AUDIO_FILE = "output_translated_audio.mp3"
# =========================

In [ ]:
# Initialize Eleven Labs client
client = ElevenLabs(api_key=ELEVENLABS_API_KEY)
print("✓ Eleven Labs client initialized\n")

# Load Whisper for English transcription
print("Loading Whisper model...")
whisper_model = whisper.load_model("small")
print("✓ Whisper loaded\n")

# Load translation model (NLLB for Telugu/Hindi)
print("Loading translation model...")
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
translation_model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
print("✓ Translation model loaded\n")

# Language codes
LANG_CODES = {
    'telugu': 'tel_Telu',
    'hindi': 'hin_Deva'
}

In [ ]:
def audio_to_segments(audio_path):
    """Split audio into segments based on silence"""
    audio = AudioSegment.from_file(audio_path)
    
    segments = split_on_silence(
        audio,
        min_silence_len=500,
        silence_thresh=audio.dBFS - 14,
        keep_silence=250
    )
    
    return segments

def transcribe_audio_segment(audio_segment):
    """Transcribe a single audio segment to English"""
    import time
    
    audio_segment = audio_segment.normalize()
    
    temp_file = f"temp_segment_{time.time()}.wav"
    audio_segment.export(
        temp_file, 
        format="wav",
        parameters=["-ar", "16000", "-ac", "1"]
    )
    
    try:
        result = whisper_model.transcribe(
            temp_file,
            language='en',
            task='transcribe',
            fp16=False
        )
        english_text = result['text'].strip()
        
        return english_text if english_text else None
                    
    except Exception as e:
        print(f"    Transcription error: {str(e)[:100]}")
        return None
                
    finally:
        try:
            time.sleep(0.1)
            if os.path.exists(temp_file):
                os.remove(temp_file)
        except:
            pass

def translate_text(text, output_language):
    """Translate English text to Telugu or Hindi"""
    if not text or not text.strip():
        return ""
    
    target_code = LANG_CODES.get(output_language.lower())
    if not target_code:
        raise ValueError(f"Language must be 'telugu' or 'hindi', got: {output_language}")
    
    tokenizer.src_lang = "eng_Latn"
    
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    
    with torch.no_grad():
        generated = translation_model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(target_code),
            max_length=1024,
            num_beams=5,
            early_stopping=True,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,
            length_penalty=1.0
        )
    
    translated_text = tokenizer.batch_decode(generated, skip_special_tokens=True)[0]
    return translated_text

def text_to_speech_elevenlabs(text, output_file, language="hindi"):
    """Convert text to speech using Eleven Labs"""
    print(f"\nGenerating audio with Eleven Labs...")
    
    # Eleven Labs voice IDs for multilingual voices
    # You can use premade voices or clone your own
    # These are examples - check Eleven Labs dashboard for available voices
    voice_id = "pNInz6obpgDQGcFmaJgB"  # Adam (multilingual)
    
    try:
        # Generate speech
        audio_generator = client.text_to_speech.convert(
            voice_id=voice_id,
            output_format="mp3_44100_128",
            text=text,
            model_id="eleven_multilingual_v2",  # Supports Hindi, Telugu, and many languages
            voice_settings=VoiceSettings(
                stability=0.5,
                similarity_boost=0.75,
                style=0.0,
                use_speaker_boost=True
            )
        )
        
        # Save audio to file
        with open(output_file, "wb") as f:
            for chunk in audio_generator:
                f.write(chunk)
        
        print(f"✓ Audio saved to: {output_file}")
        return output_file
        
    except Exception as e:
        print(f"✗ Eleven Labs error: {str(e)}")
        return None

print("Functions loaded!")

In [ ]:
# Main Processing Pipeline
print(f"Processing audio file: {AUDIO_FILE}\n")

# Step 1: Split audio into segments
segments = audio_to_segments(AUDIO_FILE)
print(f"Found {len(segments)} segments\n")

# Step 2: Transcribe all segments
all_transcripts = []
for i, segment in enumerate(segments):
    print(f"Processing segment {i+1}/{len(segments)}...")
    english_text = transcribe_audio_segment(segment)
    
    if english_text:
        print(f"  English: {english_text}")
        all_transcripts.append(english_text)
    else:
        print(f"  No speech detected")
    print()

# Step 3: Combine and translate
complete_english = " ".join(all_transcripts)
print(f"\nTranslating to {OUTPUT_LANGUAGE.capitalize()}...")
translated_text = translate_text(complete_english, OUTPUT_LANGUAGE)
print(f"✓ Translation complete\n")

# Step 4: Convert to speech with Eleven Labs
audio_output = text_to_speech_elevenlabs(translated_text, OUTPUT_AUDIO_FILE, OUTPUT_LANGUAGE)

# Display results
print("\n" + "=" * 80)
print("RESULTS")
print("=" * 80)
print(f"\nEnglish Transcript:\n{complete_english}\n")
print(f"\n{OUTPUT_LANGUAGE.capitalize()} Translation:\n{translated_text}\n")
if audio_output:
    print(f"\n✓ Translated audio saved to: {audio_output}")
else:
    print("\n✗ Audio generation failed")